# Детекция ботов по истории событий

Цель — для каждого `cookie_id` оценить вероятность того, что пользователь является ботом.  
Качество измеряется метрикой **Precision@Recall ≥ 0.70** из `metric.py`.


После серии экспериментов оставлен компактный набор из **14 интерпретируемых признаков**:
поведение во времени, работа с указателем, разнообразие просматриваемого контента,
последовательности действий и стабильность технического окружения.

Основной принцип валидации — **разделение по времени**, потому что тестовый период расположен
после обучающего.


## 1. Импорты и загрузка данных

`train` и `test` содержат по одной строке на cookie и задают окно наблюдения.
`events` — журнал событий. Признаки строятся только по событиям,
попавшим внутрь соответствующего окна.


In [2]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from metric import precision_at_recall


In [4]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])

test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])

events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print('train :', train.shape)
print('test  :', test.shape)
print('events:', events.shape)
print('доля ботов в train:', round(train['target'].mean(), 4))


train : (11091, 5)
test  : (4909, 4)
events: (328905, 14)
доля ботов в train: 0.0811


### Что видно сразу

В исходном наборе 11 091 train-cookie, 4 909 test-cookie и 328 905 событий.
Класс несбалансирован: доля ботов в train около **8.1%**.

В `events` есть как поведенческие поля (`event_name`, `search_page`, координаты указателя),
так и технические (`platform`, `user_agent`). Часть колонок заполнена только для определённых
типов событий, поэтому пропуски здесь сами по себе не являются ошибкой.


In [5]:
display(events.head())

print('\nТипы событий:')
display(events['event_name'].value_counts().to_frame('count'))

print('\nПлатформы:')
display(events['platform'].value_counts().to_frame('count'))

print('\nДоля пропусков:')
display(events.isna().mean().sort_values(ascending=False).round(3).to_frame('missing_share'))


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN



Типы событий:


,count
event_name,
item_view,120817
search_results_view,100402
photo_swipe,36517
favorite_add,19049
seller_page_view,17403
contact_phone_show,11316
captcha_shown,7928
login,6267
contact_chat_open,6125



Платформы:


,count
platform,
desktop,46866
WEB,46476
Web,46365
web,45951
ANDROID,42462
Android,42254
android,42159
iphone,4103
IOS,4100



Доля пропусков:


,missing_share
search_query,0.695
search_page,0.695
pointer_x,0.670
pointer_y,0.670
seller_type,0.407
item_id,0.348
item_category,0.100
item_location,0.072
user_agent,0.000
platform,0.000


## 2. Подготовка событий

Есть два важных правила.

1. Используются только события из окна  
   `window_start_ts <= event_ts < window_end_ts`.
2. Удаляются только **полные дубликаты строк**.  
   События с одинаковыми `cookie_id + event_ts + eid`, но разными остальными полями
   могут быть реальными разными событиями, поэтому по этому ключу я строки не схлопываю.

Для одинакового порядка событий при совпадающих timestamp использую `eid` как tie-breaker.


In [6]:
def prepare_events(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    meta_local = meta.copy()

    meta_local['window_start_ts'] = pd.to_datetime(meta_local['window_start_ts'])
    meta_local['window_end_ts'] = pd.to_datetime(meta_local['window_end_ts'])

    ev = events.copy()
    ev['event_ts'] = pd.to_datetime(ev['event_ts'])

    # Защита от повторного запуска ячеек.
    ev = ev.drop(
        columns=['window_start_ts', 'window_end_ts'],
        errors='ignore',
    )

    ev = ev.merge(
        meta_local[['cookie_id', 'window_start_ts', 'window_end_ts']],
        on='cookie_id',
        how='inner',
        validate='many_to_one',
    )

    ev = ev[
        (ev['event_ts'] >= ev['window_start_ts'])
        & (ev['event_ts'] < ev['window_end_ts'])
    ].copy()

    ev = ev.drop_duplicates()

    return ev.sort_values(
        ['cookie_id', 'event_ts', 'eid'],
        kind='mergesort',
    ).reset_index(drop=True)


In [7]:
ev_train = prepare_events(events, train)
ev_test = prepare_events(events, test)

print('train events:', len(ev_train))
print('test events :', len(ev_test))
print('полных дублей после очистки:', ev_train.duplicated().sum())


train events: 195484
test events : 88349
полных дублей после очистки: 0


В исходных train-событиях было **2 952 полных дубля**. Их удаление безопасно,
потому что они не несут новой информации. При этом строки с совпадающим техническим ключом,
но отличающимся содержимым, сохраняются.


## 3. Временная валидация

Train заканчивается 19 апреля, а test начинается 20 апреля. Поэтому случайный split
был бы слишком оптимистичным: модель училась бы на «будущем» относительно части валидации.

Основная проверка — последние **7 дней train** (`2026-04-13` … `2026-04-19`).
Это особенно удобно, потому что test также покрывает 7 дней (`2026-04-20` … `2026-04-26`).

Дополнительно использую последние **3 дня** как вторую проверку устойчивости.
Решение в первую очередь выбираю по 7-дневному сплиту.


In [8]:
print('train:', train['window_start_ts'].min(), '—', train['window_start_ts'].max())
print('test :', test['window_start_ts'].min(), '—', test['window_start_ts'].max())

VALIDATION_CUTOFFS = {
    '7 дней': pd.Timestamp('2026-04-13'),
    '3 дня': pd.Timestamp('2026-04-17'),
}

y = train['target'].to_numpy()
dates = pd.to_datetime(train['window_start_ts'])


train: 2026-04-06 00:00:00 — 2026-04-19 00:00:00
test : 2026-04-20 00:00:00 — 2026-04-26 00:00:00


In [9]:
def evaluate_model(model, X, columns):
    rows = []

    for split_name, cutoff in VALIDATION_CUTOFFS.items():
        valid_mask = dates.ge(cutoff).to_numpy()

        fitted = clone(model)
        fitted.fit(
            X.loc[~valid_mask, columns],
            y[~valid_mask],
        )

        pred = fitted.predict_proba(
            X.loc[valid_mask, columns]
        )[:, 1]

        rows.append({
            'validation': split_name,
            'train_size': int((~valid_mask).sum()),
            'valid_size': int(valid_mask.sum()),
            'P@R0.7': precision_at_recall(y[valid_mask], pred),
        })

    return pd.DataFrame(rows)


## 4. Baseline

Сначала нужен простой ориентир. В baseline использую только количество событий
и число уникальных объявлений. Это специально слабая модель: её задача — показать,
что дальнейший прирост действительно создаётся feature engineering, а не случайностью.


In [10]:
def make_basic_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    g = ev.groupby('cookie_id')

    features = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g['item_id'].nunique(),
    })

    return (
        meta[['cookie_id']]
        .merge(features.reset_index(), on='cookie_id', how='left', validate='one_to_one')
        .fillna(0)
    )


In [11]:
X_basic = make_basic_features(ev_train, train)
basic_cols = ['n_events', 'item_nunique']

baseline = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0,
    n_jobs=-1,
)

display(evaluate_model(baseline, X_basic, basic_cols).round(4))


,validation,train_size,valid_size,P@R0.7
0,7 дней,5930,5161,0.0931
1,3 дня,9140,1951,0.0957


Baseline показывает, что простых счётчиков недостаточно. Поэтому дальше я перехожу
к признакам, описывающим **характер поведения**, а не только объём активности.


## 5. Финальные признаки

Финальный набор состоит из пяти небольших блоков.

| Блок | Идея |
|---|---|
| Время | насколько часто события идут быстро и как устроены сессии |
| Указатель | насколько разнообразно пользователь двигает курсор |
| Контент | насколько разнообразны категории, объявления и локации |
| Последовательности | как часто два `item_view` идут подряд |
| User-Agent | насколько стабильно техническое окружение cookie |

Большой исходный набор признаков я не оставляю: на временной валидации компактные признаки
работали лучше и проще интерпретируются.


In [12]:
def entropy_feature(series: pd.Series) -> float:
    counts = series.dropna().value_counts()

    if counts.empty:
        return 0.0

    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum())


### 5.1. Временные признаки

- `delta_le_60_share` — доля соседних событий с паузой не больше минуты.
  Это устойчивый сигнал интенсивности без привязки к абсолютному числу событий.
- `sessions_30m_mean` — среднее число событий в сессии, где новая сессия начинается
  после паузы более 30 минут.
- `created_recent_7d` — новая ли cookie относительно начала окна наблюдения.


In [13]:
def make_time_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    base = meta.set_index('cookie_id').copy()
    out = pd.DataFrame(index=base.index)

    age_seconds = (
        pd.to_datetime(base['window_start_ts'])
        - pd.to_datetime(base['cookie_created_at'])
    ).dt.total_seconds()

    out['created_recent_7d'] = (age_seconds < 7 * 86400).astype(int)

    work = ev[['cookie_id', 'event_ts']].copy()
    work['delta_sec'] = (
        work.groupby('cookie_id')['event_ts']
        .diff()
        .dt.total_seconds()
    )

    valid_delta = work.dropna(subset=['delta_sec'])

    out['delta_le_60_share'] = (
        valid_delta['delta_sec']
        .le(60)
        .groupby(valid_delta['cookie_id'])
        .mean()
    )

    new_session = work['delta_sec'].isna() | work['delta_sec'].gt(1800)
    session_id = new_session.groupby(work['cookie_id']).cumsum()

    session_sizes = work.groupby(
        [work['cookie_id'], session_id]
    ).size()

    out['sessions_30m_mean'] = (
        session_sizes.groupby(level=0).mean()
    )

    return out


### 5.2. Признаки указателя

`pointer_x_std` оказался одним из самых сильных признаков в ablation-тесте:
если его убрать, качество заметно падает.  
Дополнительно `pointer_y_range` дал небольшой, но стабильный прирост как отдельный признак.


In [14]:
def make_pointer_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    g = ev.groupby('cookie_id')

    out = pd.DataFrame(
        index=meta.set_index('cookie_id').index
    )

    out['pointer_x_std'] = g['pointer_x'].std()

    out['pointer_y_range'] = (
        g['pointer_y'].max()
        - g['pointer_y'].min()
    )

    return out


### 5.3. Разнообразие контента и глубина поиска

Энтропия используется вместо простого числа уникальных значений, потому что она учитывает
не только количество вариантов, но и то, насколько равномерно они встречаются.

- `category_entropy`, `item_entropy`, `location_entropy` — разнообразие интересов;
- `category_unique_ratio` — доля уникальных категорий среди непустых наблюдений;
- `search_page_value_q90` — высокая квантиль номера страницы поиска, то есть типичная
  «глубина» просмотра без чрезмерной чувствительности к единичному максимуму.


In [15]:
def make_content_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    g = ev.groupby('cookie_id')

    out = pd.DataFrame(
        index=meta.set_index('cookie_id').index
    )

    category_count = g['item_category'].count().astype(float)
    category_nunique = g['item_category'].nunique().astype(float)

    out['category_unique_ratio'] = (
        category_nunique
        / category_count.clip(lower=1)
    )

    out['category_entropy'] = g['item_category'].apply(entropy_feature)
    out['location_entropy'] = g['item_location'].apply(entropy_feature)
    out['item_entropy'] = g['item_id'].apply(entropy_feature)

    out['search_page_value_q90'] = (
        g['search_page'].quantile(0.90)
    )

    return out


### 5.4. Последовательность действий

Из нескольких переходов событий полезным оказался только `item_view → item_view`.
Признак `tr_item_item` — доля таких переходов среди всех соседних пар событий cookie.

Идея: автоматизированный просмотр карточек может давать другой паттерн переходов,
чем обычная пользовательская навигация.


In [16]:
def make_sequence_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    work = ev[['cookie_id', 'event_name']].copy()

    work['prev_event'] = (
        work.groupby('cookie_id')['event_name']
        .shift()
    )

    transitions = work.dropna(subset=['prev_event'])

    is_item_to_item = (
        (transitions['prev_event'] == 'item_view')
        & (transitions['event_name'] == 'item_view')
    )

    out = pd.DataFrame(
        index=meta.set_index('cookie_id').index
    )

    out['tr_item_item'] = (
        is_item_to_item
        .groupby(transitions['cookie_id'])
        .mean()
    )

    return out


### 5.5. Разнообразие User-Agent

Сырые строки `user_agent` не подаются в модель. Вместо этого использую только три
понятных агрегата:

- `ua_browser_nunique` — число разных браузерных семейств;
- `ua_platform_nunique` — число разных нормализованных платформ;
- `ua_nunique` — число разных полных User-Agent.

По отдельности эти признаки давали небольшой эффект, а вместе улучшили оба временных split,
поэтому я оставляю именно группу **UA diversity**, без дополнительных headless/app-флагов.


In [17]:
def make_ua_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    work = ev[['cookie_id', 'platform', 'user_agent']].copy()

    ua = work['user_agent'].fillna('').astype(str).str.lower()

    work['ua_browser'] = np.select(
        [
            ua.str.contains('headlesschrome', regex=False),
            ua.str.contains('yabrowser', regex=False),
            ua.str.contains('firefox', regex=False),
            ua.str.contains('chrome', regex=False),
        ],
        ['headless', 'yandex', 'firefox', 'chrome'],
        default='other',
    )

    work['platform_norm'] = (
        work['platform']
        .fillna('missing')
        .astype(str)
        .str.strip()
        .str.lower()
    )

    g = work.groupby('cookie_id')

    out = pd.DataFrame(
        index=meta.set_index('cookie_id').index
    )

    out['ua_browser_nunique'] = g['ua_browser'].nunique()
    out['ua_platform_nunique'] = g['platform_norm'].nunique()
    out['ua_nunique'] = g['user_agent'].nunique()

    return out


### 5.6. Сборка матрицы признаков

Каждый блок строится независимо, после чего объединяется по `cookie_id`.
Так код проще проверять и менять: один эксперимент не затрагивает остальные части pipeline.


In [18]:
def make_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    parts = [
        make_time_features(ev, meta),
        make_pointer_features(ev, meta),
        make_content_features(ev, meta),
        make_sequence_features(ev, meta),
        make_ua_features(ev, meta),
    ]

    features = pd.concat(parts, axis=1)

    if features.columns.duplicated().any():
        repeated = features.columns[features.columns.duplicated()].tolist()
        raise ValueError(f'Повторяющиеся признаки: {repeated}')

    features.index.name = 'cookie_id'

    return (
        features
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .reset_index()
    )


## 6. Отбор компактного набора

Сначала я проверила 13 интерпретируемых кандидатов. Затем сделала leave-one-feature-out
ablation: по очереди убирала один признак и смотрела, меняется ли качество на двух временных split.

Четыре признака не давали стабильного выигрыша и были исключены:
`delta_median`, `delta_le_30_share`, `delta_unique_ratio`,
`events_per_active_minute`.

После этого осталось ядро из 9 признаков. Для Random Forest оно оказалось лучше
расширенного набора на основной 7-дневной валидации:

| Набор | 3 дня | 7 дней |
|---|---:|---:|
| 13 признаков | 0.6667 | 0.5755 |
| 9 признаков | 0.6474 | **0.6093** |



In [19]:
CORE9 = [
    'pointer_x_std',
    'delta_le_60_share',
    'category_entropy',
    'created_recent_7d',
    'search_page_value_q90',
    'sessions_30m_mean',
    'item_entropy',
    'category_unique_ratio',
    'location_entropy',
]

CORE10 = CORE9 + [
    'pointer_y_range',
]

BASE11 = CORE10 + [
    'tr_item_item',
]

FINAL14 = BASE11 + [
    'ua_browser_nunique',
    'ua_platform_nunique',
    'ua_nunique',
]


In [20]:
X_train = make_features(ev_train, train)
X_test = make_features(ev_test, test)

assert X_train['cookie_id'].reset_index(drop=True).equals(
    train['cookie_id'].reset_index(drop=True)
)

assert X_test['cookie_id'].reset_index(drop=True).equals(
    test['cookie_id'].reset_index(drop=True)
)

assert X_train[FINAL14].isna().sum().sum() == 0
assert X_test[FINAL14].isna().sum().sum() == 0

print('train features:', X_train[FINAL14].shape)
print('test features :', X_test[FINAL14].shape)
print('число финальных признаков:', len(FINAL14))


train features: (11091, 14)
test features : (4909, 14)
число финальных признаков: 14


## 7. Выбор модели

На одинаковом ядре из 9 признаков я сравнила несколько простых моделей.
Логистическая регрессия заметно проиграла — сигнал явно нелинейный.
Деревья работали намного лучше, а лучший результат дал
`HistGradientBoostingClassifier`.

Полученные значения:

| Модель | 3 дня | 7 дней |
|---|---:|---:|
| HistGradientBoosting | **0.6726** | **0.6560** |
| RF balanced | 0.6667 | 0.6172 |
| RandomForest | 0.6512 | 0.5971 |
| ExtraTrees | 0.6552 | 0.5946 |
| LogisticRegression | 0.2358 | 0.2201 |

Поэтому финальная модель — градиентный бустинг по небольшим деревьям.


In [21]:
models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=700,
        min_samples_leaf=8,
        random_state=0,
        n_jobs=-1,
    ),

    'RF balanced': RandomForestClassifier(
        n_estimators=700,
        min_samples_leaf=8,
        class_weight='balanced_subsample',
        random_state=0,
        n_jobs=-1,
    ),

    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=700,
        min_samples_leaf=8,
        max_features=0.8,
        random_state=0,
        n_jobs=-1,
    ),

    'HistGradientBoosting': HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=5,
        random_state=0,
    ),

    'LogisticRegression': make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight='balanced',
            max_iter=2000,
            random_state=0,
        ),
    ),
}


In [22]:
model_rows = []

for model_name, candidate in models.items():
    scores = evaluate_model(candidate, X_train, CORE9)

    for row in scores.to_dict('records'):
        model_rows.append({
            'model': model_name,
            **row,
        })

model_comparison = pd.DataFrame(model_rows)

display(
    model_comparison
    .pivot(index='model', columns='validation', values='P@R0.7')
    .round(4)
    .sort_values('7 дней', ascending=False)
)


validation,3 дня,7 дней
model,,
HistGradientBoosting,0.6871,0.6330
RF balanced,0.6686,0.6207
RandomForest,0.6474,0.6021
ExtraTrees,0.6627,0.5946
LogisticRegression,0.2358,0.2201


## 8. Как собирался финальный набор

После выбора HGB я добавляла новые блоки только если они улучшали временную валидацию.

| Набор | 3 дня | 7 дней | Решение |
|---|---:|---:|---|
| CORE9 | 0.6726 | 0.6560 | базовое компактное ядро |
| + `pointer_y_range` | **0.6957** | 0.6590 | оставить |
| + `tr_item_item` | 0.6826 | 0.6729 | оставить: сильный рост на основном split |
| + UA diversity | **0.7044** | **0.6850** | оставить |

Именно поэтому в финале 14 признаков, а не все проверенные идеи.

Неудачные гипотезы — регулярность временных интервалов, длины серий `item_view`,
разнообразие поисковых запросов и дополнительные характеристики перемещений указателя —
в финальную модель не входят: на 7-дневной проверке они не улучшали результат.


In [23]:
final_hgb = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=300,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=5,
    random_state=0,
)

feature_sets = {
    'CORE9': CORE9,
    '+ pointer_y_range': CORE10,
    '+ tr_item_item': BASE11,
    '+ UA diversity': FINAL14,
}

feature_rows = []

for set_name, columns in feature_sets.items():
    scores = evaluate_model(final_hgb, X_train, columns)

    for row in scores.to_dict('records'):
        feature_rows.append({
            'features': set_name,
            **row,
        })

feature_comparison = pd.DataFrame(feature_rows)

display(
    feature_comparison
    .pivot(index='features', columns='validation', values='P@R0.7')
    .round(4)
    .loc[list(feature_sets)]
)


validation,3 дня,7 дней
features,,
CORE9,0.6871,0.6330
+ pointer_y_range,0.6957,0.6341
+ tr_item_item,0.7019,0.6493
+ UA diversity,0.6829,0.6457


## 9. Финальная модель

Гиперпараметры HGB подбирались небольшим ручным сравнением.
Более глубокие деревья и изменение регуляризации не дали выигрыша,
поэтому оставлена простая конфигурация:

- `learning_rate=0.04`;
- `max_iter=300`;
- `max_leaf_nodes=15`;
- `min_samples_leaf=20`;
- `l2_regularization=5`.

После выбора признаков модель переобучается на **всём train**.


In [24]:
final_model = HistGradientBoostingClassifier(
    learning_rate=0.04,
    max_iter=300,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=5,
    random_state=0,
)

final_validation = evaluate_model(
    final_model,
    X_train,
    FINAL14,
)

display(final_validation.round(4))


,validation,train_size,valid_size,P@R0.7
0,7 дней,5930,5161,0.6457
1,3 дня,9140,1951,0.6829


In [25]:
final_model.fit(
    X_train[FINAL14],
    y,
)

test_score = final_model.predict_proba(
    X_test[FINAL14]
)[:, 1]


## 10. Submission

Перед сохранением проверяю четыре вещи:

- число строк совпадает с test;
- `cookie_id` уникальны;
- порядок cookie не изменился;
- все score конечны и лежат в `[0, 1]`.


In [26]:
submission = pd.DataFrame({
    'cookie_id': test['cookie_id'],
    'score': test_score,
})

assert len(submission) == len(test)
assert submission['cookie_id'].is_unique
assert submission['cookie_id'].equals(test['cookie_id'])
assert submission['score'].between(0, 1).all()
assert submission['score'].notna().all()

submission.to_csv(
    'submission_final.csv',
    index=False,
)

print('rows:', len(submission))
print('score min:', submission['score'].min())
print('score max:', submission['score'].max())

display(submission.head())


rows: 4909
score min: 0.001331330586261306
score max: 0.9750632642721977


,cookie_id,score
0,ck_315fb710a0e371e7,0.001728
1,ck_a76ee3b3e3e522fd,0.149450
2,ck_94c9a4d382689e82,0.028307
3,ck_8eaf9509ad9462a0,0.002315
4,ck_9a88a5a989cb5bc6,0.017894


## 11. Что означает каждый из 14 финальных признаков

| Признак | Смысл |
|---|---|
| `pointer_x_std` | разброс координаты X указателя |
| `delta_le_60_share` | доля соседних событий с паузой ≤ 60 секунд |
| `category_entropy` | разнообразие категорий с учётом частот |
| `created_recent_7d` | cookie создана менее чем за 7 дней до окна |
| `search_page_value_q90` | 90-й перцентиль глубины поисковой выдачи |
| `sessions_30m_mean` | среднее число событий в 30-минутной сессии |
| `item_entropy` | разнообразие просматриваемых объявлений |
| `category_unique_ratio` | доля уникальных категорий |
| `location_entropy` | разнообразие локаций |
| `pointer_y_range` | диапазон перемещения указателя по Y |
| `tr_item_item` | доля переходов `item_view → item_view` |
| `ua_browser_nunique` | число браузерных семейств внутри cookie |
| `ua_platform_nunique` | число нормализованных платформ |
| `ua_nunique` | число разных полных User-Agent |


## 12. Итог

Финальное решение получилось компактным: **14 признаков + HistGradientBoosting**.

Главный прирост дал переход от простого Random Forest к градиентному бустингу
и отбор небольшого набора поведенческих признаков. UA diversity дал уже небольшой
дополнительный прирост.

Ограничение решения: признаки и гиперпараметры несколько раз проверялись на одних и тех же
временных split, поэтому локальная метрика немного оптимистична. Именно поэтому после
получения компактной устойчивой конфигурации дальнейший перебор был остановлен.
